In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
from scipy.optimize import minimize
import calendar

In [2]:
df = pd.read_excel("Assignment 3 Data-1.xlsx", sheet_name = None)
type(df)

dict

In [3]:
df_run = df["Run"]
df_yc = df["YieldCurve"]

In [4]:
df_yc["Date"] = pd.to_datetime(df_yc["Date"], errors="coerce")
df_yc["Maturity"] = pd.to_datetime(df_yc["Maturity"], errors="coerce")

In [5]:
## Tenor calculation
def Tenor(row):
    start = row["Date"]
    maturity = row["Maturity"]
    num = (360 * (maturity.year - start.year)
           + 30 * (maturity.month - start.month)
           + (maturity.day - start.day))
    return num / 360

In [6]:
df_yc["Tenor"] = df_yc.apply(lambda row: Tenor(row), axis=1)

## PART 1

In [8]:
## Nss spot rate curve equation
def nss_spot(t, beta0, beta1, beta2, beta3, tau1, tau2):
    t = np.asarray(t, dtype=float)
    def f(t, tau):
        x = t / tau
        # avoiding division by zero
        return (1 - np.exp(-x)) / np.where(x == 0, 1e-8, x)
    f1 = f(t, tau1)
    f2 = f(t, tau2)
    r = (beta0 + beta1 * f1 + beta2 * (f1 - np.exp(-t / tau1)) + beta3 * (f2 - np.exp(-t / tau2)))
    return r

In [9]:
## obejctive function for scipy optimize
def nss_objective(params, tenor, yield_):
    beta0, beta1, beta2, beta3, tau1, tau2 = params
    model = nss_spot(tenor, beta0, beta1, beta2, beta3, tau1, tau2)
    resid = model - yield_
    return np.sum(resid**2)

In [10]:
# Initial guess (Betas and Taus)
x0 = np.array([0.03, 0.01, -0.03, 0.05, 1.0, 20.0])

In [11]:
df_yc_yield = df_yc["Yield"]/100.0
res = minimize(nss_objective, x0, args=(df_yc["Tenor"], df_yc_yield), method="BFGS")
print("Success:", res.success)
print("Params:", res.x)
print("Current Function Value (SSR < 0.5):", res.fun)

Success: True
Params: [2.62007834e-02 1.24615467e-02 2.15608359e-03 7.49319521e-02
 1.00124127e+00 1.99999885e+01]
Current Function Value (SSR < 0.5): 6.496902291006065e-06


In [12]:
## assigning param
beta0, beta1, beta2, beta3, tau1, tau2 = res.x

In [13]:
def generate_coupon_dates(start, maturity):
    dates = []
    current = start
    while True:
        y, m = current.year, current.month
        # move forward 1 month
        if m == 12:
            y += 1
            m = 1
        else:
            m += 1
        # 30/360 rule: day = 30 unless month has <30 days
        last_day = calendar.monthrange(y, m)[1]
        day = min(30, last_day)
        d = pd.Timestamp(year=y, month=m, day=day)
        dates.append(d)
        if d >= maturity:
            break
        current = d
    return pd.DatetimeIndex(dates)

In [14]:
def price_bond(row, risk = 0):
    face = 100.0   ##assuming all bonds ahve face value of 100 unless given
    start = row["Date"]
    maturity = row["Maturity"]
    coupon_rate = row["Coupon"] / 100.0
    z_spread = row["Z Spread"] / 10000.0

    pay_dates = generate_coupon_dates(start, maturity)
    
    pv = 0
    prev = start
    for d in pay_dates:
        t = Tenor(pd.Series({"Date": start, "Maturity": d}))
        dtau = Tenor(pd.Series({"Date": prev, "Maturity": d}))
        spot = nss_spot(t, beta0, beta1, beta2, beta3, tau1, tau2)

        ## adding spot + spread + risk 
        r = spot + z_spread + risk

        ## DF calc
        df = 1.0 / (1.0 + r)**t

        ## IMP: cash-flow = coupon_rate * dtau * face
        cf = face * coupon_rate * dtau
        if d == pay_dates[-1]:
            cf += face
 
        pv += cf * df
        prev = d
    return pv

In [15]:
dy = 0
df_run["Price"] = df_run.apply(lambda row : price_bond(row, risk=dy), axis = 1)

In [16]:
df_run[df_run["Ticker"] == "ABT"]["Price"].round(2)

0    89.64
Name: Price, dtype: float64

In [17]:
df_run

,Ticker,Coupon,Maturity,Sector,Credit Rating,Z Spread,Date,CUSIP,Price
0,ABT,1.400,2030-06-30,Healthcare,AA,23,2025-11-06,002824BQ2,89.637929
1,APH,5.000,2035-01-15,Technology & Electronics,A,72,2025-11-06,032095AR2,102.817657
2,BA,3.250,2028-03-01,Capital Goods,BBB,68,2025-11-06,097023BX2,97.899769
3,F,6.532,2032-03-19,Automotive,BBB,191,2025-11-06,345397G98,105.235653
4,IBM,4.600,2027-02-05,Technology & Electronics,A,35,2025-11-06,449276AB0,100.900737
5,LOW,5.150,2033-07-01,Retail,BBB,75,2025-11-06,548661EQ6,103.902210
6,META,4.300,2029-08-15,Media,AA,36,2025-11-06,30303M8S4,101.387068
7,NFLX,4.375,2026-11-15,Media,A,23,2025-11-06,64110LAN6,100.596891
8,TMO,5.200,2034-01-31,Healthcare,A,62,2025-11-06,883556DB5,105.090979
9,TMUS,3.875,2030-04-15,Telecommunications,BBB,68,2025-11-06,87264ABF1,98.446592


## PART 2

In [19]:
dy = 0.0001
P0 = df_run.apply(lambda row : price_bond(row, risk=0), axis = 1)
P_up = df_run.apply(lambda row : price_bond(row, risk=dy), axis = 1)
P_down = df_run.apply(lambda row : price_bond(row, risk=-dy), axis = 1)

In [20]:
## Effective Duration
df_run["Duration"] = (P_down - P_up)/(2 * P0 * dy)

In [21]:
# Effective Convexity
df_run["Convexity"] = (P_up + P_down - 2 * P0)/(P0 * dy**2)/100.0

In [22]:
df_run

,Ticker,Coupon,Maturity,Sector,Credit Rating,Z Spread,Date,CUSIP,Price,Duration,Convexity
0,ABT,1.400,2030-06-30,Healthcare,AA,23,2025-11-06,002824BQ2,89.637929,4.324827,0.232941
1,APH,5.000,2035-01-15,Technology & Electronics,A,72,2025-11-06,032095AR2,102.817657,7.094946,0.645286
2,BA,3.250,2028-03-01,Capital Goods,BBB,68,2025-11-06,097023BX2,97.899769,2.216801,0.071627
3,F,6.532,2032-03-19,Automotive,BBB,191,2025-11-06,345397G98,105.235653,4.995374,0.329164
4,IBM,4.600,2027-02-05,Technology & Electronics,A,35,2025-11-06,449276AB0,100.900737,1.226694,0.027114
5,LOW,5.150,2033-07-01,Retail,BBB,75,2025-11-06,548661EQ6,103.902210,6.134495,0.482237
6,META,4.300,2029-08-15,Media,AA,36,2025-11-06,30303M8S4,101.387068,3.394436,0.153859
7,NFLX,4.375,2026-11-15,Media,A,23,2025-11-06,64110LAN6,100.596891,1.005514,0.019926
8,TMO,5.200,2034-01-31,Healthcare,A,62,2025-11-06,883556DB5,105.090979,6.506015,0.542761
9,TMUS,3.875,2030-04-15,Telecommunications,BBB,68,2025-11-06,87264ABF1,98.446592,3.947690,0.202502
